# AgentRegistry, end to end: scaffold a dice agent, add approved MCP tools, deploy to kagent **and** AWS Bedrock AgentCore, then govern the tools

A developer builds an agent with `arctl`, runs it locally, pulls in **approved MCP tool servers** from the registry catalog, publishes the agent, and deploys the *same* published agent to two runtimes (Solo Enterprise for kagent on kind, and AWS Bedrock AgentCore). Finally a platform owner restricts which MCP tools the agent may call with an **AccessPolicy**, enforced at an agentgateway waypoint.

> **Kernel:** pick **Bash** (top-right). One-time engineer setup lives in `scripts/` (see `README.md`).

## Open the consoles

Run this once **in a terminal** — it port-forwards the kagent UI and opens both tabs (no `/etc/hosts` edit needed):

```sh
./scripts/open-consoles.sh
```

| Console | URL | Login |
|---|---|---|
| AgentRegistry UI | http://localhost:12121 | — |
| kagent UI | http://localhost:18007 | alice / alice |
| AWS Bedrock AgentCore | the AWS console (manual) | — |

## Connect to the platform

Loads creds, puts `arctl` on the path, mints a catalog token.

In [ ]:
[ -d agentregistry-agentcore-kind ] && cd agentregistry-agentcore-kind || :; source scripts/connect.sh && rm -rf agentdemo

## 1. Browse the approved tool-server catalog

Rather than wiring arbitrary MCP servers, developers pull from an **approved catalog**. Open the **AgentRegistry UI** (http://localhost:12121) → **Tool Servers** to see `everything-server` and `my-mcp`. The same list from the CLI:

In [ ]:
arctl get mcpservers

## 2. Create the agent, wired to the approved tools

Scaffold the agent and reference the approved servers straight from the catalog with `--mcp` (repeatable). arctl records them in `agent.yaml` under `spec.mcpServers`, resolved into tool servers beside the agent at deploy time. The generated agent also has two **local** tools, `roll_die` and `check_prime`.

In [ ]:
arctl init agent agentdemo --framework adk --language python --model-provider anthropic --model-name claude-haiku-4-5 --mcp demo/everything-server@latest --mcp demo/my-mcp@latest

In [ ]:
cat agentdemo/agentdemo/agent.py

In [ ]:
cat agentdemo/agent.yaml

## 3. Add the approved dice-game skill

A skill is behavioural guidance, not a tool. Pull it from the catalog and fold its body into the agent's system instruction. The Agent kind has no `skills` field (and there is no `--skill` flag), so the guidance lives in `build_instruction(...)` in `agent.py`.

In [ ]:
# Pull the approved skill from the catalog (native), then fold its guidance into
# the agent's system instruction. There is no --skill flag / no spec.skills field,
# so a skill lives in build_instruction(...) in agent.py:
arctl get skill dice-game -o yaml

python3 - <<'EOF'
import re
body = re.sub(r'^---\n.*?\n---\n', '', open("skill/dice-game/SKILL.md").read(), count=1, flags=re.S).strip()
p = "agentdemo/agentdemo/agent.py"
src = open(p).read()
src = re.sub(r'build_instruction\(\s*"""(.*?)"""\s*\)',
             'build_instruction("""\n' + body.replace("\\", "\\\\") + '\n""")', src, count=1, flags=re.S)
open(p, "w").write(src)
print("\nbaked dice-game skill into", p)
EOF

## 4. Build and run it locally

Build the image, then chat with the agent **in a terminal** (`arctl run` is interactive, so not a notebook cell):

```sh
arctl run ./agentdemo
```
Ask: *"Roll a 20-sided die and tell me whether the result is prime."* It uses the local `roll_die` + `check_prime` tools and the dice-game skill's house format. The approved catalog tools (`sum`, `echo`, `word_count`, ...) come online when the agent is **deployed**: the registry stands up the MCP servers beside it (you'll see them next).

In [ ]:
arctl build ./agentdemo

## 5. Build (multi-arch), publish, and push the source

Multi-arch so the same image runs on kagent (arm64 here) and AgentCore (amd64).

In [ ]:
arctl build ./agentdemo --platform linux/amd64,linux/arm64 --push

In [ ]:
arctl apply -f agentdemo/agent.yaml

AgentCore clones the agent **source** from git at deploy time, so push it to your agent repo (the engineer set `AGENT_GIT_URL` in `.env.local`):

In [ ]:
# AgentCore clones the agent SOURCE from git at deploy time, so push agentdemo/
# to your agent repo ($AGENT_GIT_URL, from .env.local).
set -o pipefail
: "${AGENT_GIT_URL:?set AGENT_GIT_URL in .env.local}"
SLUG="${AGENT_GIT_URL#https://github.com/}"; SLUG="${SLUG%.git}"; BR="${AGENT_GIT_BRANCH:-main}"
PUSH_URL="https://x-access-token:$(gh auth token)@github.com/${SLUG}.git"
T="$(mktemp -d)"; cp -R agentdemo "$T/agentdemo"; rm -rf "$T/agentdemo/.git" "$T/agentdemo/.venv"
( cd "$T" \
  && git init -qb "$BR" && git add -A \
  && git -c user.email=demo@local -c user.name=demo commit -qm "agentdemo source" \
  && git remote add origin "$PUSH_URL" && git push -fq origin "$BR" )
rm -rf "$T"
echo "pushed agentdemo/ source -> $SLUG@$BR"

## 6. Deploy onto kagent (runtime #1)

Binds the agent to the `kind-kagent` runtime. The registry stands up the agent **and** the referenced MCP server as pods, wired together.

In [ ]:
envsubst < yaml/deploy-kagent.yaml | arctl apply -f -

In [ ]:
kubectl --context kind-$CLUSTER_NAME -n kagent get pods | grep -E 'agentdemo|everything-server|my-mcp'

## 7. Deploy the same agent to AWS Bedrock AgentCore (runtime #2)

Same published agent, different `runtimeRef`. On AgentCore it uses Bedrock Claude via the AWS role (no API key). Requires an AWS account; skip for a local-only run.

In [ ]:
source scripts/aws-login.sh

In [ ]:
# Deploy the SAME agent to AWS Bedrock AgentCore. Needs an AWS session (run the
# 'source scripts/aws-login.sh' cell first). This is the full sequence: make the
# agent multi-cloud, mint a cross-account role via CloudFormation, register the
# runtime, push image + source, then arctl apply the Agent and the Deployment.
set -o pipefail
export AWS_REGION="${AWS_REGION:-us-east-1}"
AGENT=agentdemo; STACK="${STACK_NAME:-AgentRegistryAccess}"

echo "▸ Preflight"
aws sts get-caller-identity >/dev/null 2>&1 || { echo "no AWS session — run the aws-login cell"; exit 1; }
AWS_ACCOUNT_ID="$(aws sts get-caller-identity --query Account --output text)"
: "${AGENT_GIT_URL:?set AGENT_GIT_URL in .env.local}"
echo "  AWS $AWS_ACCOUNT_ID / $AWS_REGION"

echo "▸ Make the agent multi-cloud (Bedrock model via MODEL_PROVIDER)"
cp templates/bedrock_model.py agentdemo/agentdemo/bedrock_model.py
python3 - agentdemo <<'PY'
import re, sys, pathlib
proj = pathlib.Path(sys.argv[1])
p = proj/'agentdemo'/'agent.py'; s = p.read_text()
if 'MODEL_PROVIDER' not in s:
    s = s.replace('from google.adk.models.lite_llm import LiteLlm\n', '')
    new = ('def create_model():\n'
           '    import os\n'
           '    if os.environ.get("MODEL_PROVIDER", "anthropic").lower() == "bedrock":\n'
           '        from .bedrock_model import BedrockClaude\n'
           '        return BedrockClaude(model=os.environ.get("MODEL_NAME", "us.anthropic.claude-haiku-4-5-20251001-v1:0"))\n'
           '    from google.adk.models.lite_llm import LiteLlm\n'
           '    return LiteLlm(model=os.environ.get("MODEL_NAME", "anthropic/claude-haiku-4-5"))\n')
    s = re.sub(r'def create_model\(\):.*?(?=\n\nroot_agent|\nroot_agent|\Z)', new.rstrip()+'\n', s, count=1, flags=re.S)
    p.write_text(s)
pp = proj/'pyproject.toml'; t = pp.read_text()
if 'anthropic[bedrock]' not in t:
    pp.write_text(t.replace('dependencies = [', 'dependencies = [\n  "anthropic[bedrock]>=0.40",', 1))
print("  multi-cloud patch applied")
PY

echo "▸ Cross-account role (arctl runtime setup -> CloudFormation)"
mkdir -p .agentcore
arctl runtime setup bedrock-agent-core --aws-account-id "$AWS_ACCOUNT_ID" \
  --role-name "AgentRegistryAccessRole-agentcore-demo" \
  > .agentcore/cf.yaml 2> .agentcore/setup.stderr || { cat .agentcore/setup.stderr; echo "runtime setup failed"; exit 1; }
AWS_EXTERNAL_ID="$(grep -ioE 'External ID:[[:space:]]*[A-Za-z0-9_-]+' .agentcore/setup.stderr | awk '{print $NF}' | head -1)"
[ -n "$AWS_EXTERNAL_ID" ] || { echo "could not parse External ID"; exit 1; }
if aws cloudformation describe-stacks --stack-name "$STACK" >/dev/null 2>&1; then
  echo "  stack $STACK exists"
else
  aws cloudformation create-stack --stack-name "$STACK" --template-body "file://.agentcore/cf.yaml" --capabilities CAPABILITY_NAMED_IAM >/dev/null
  aws cloudformation wait stack-create-complete --stack-name "$STACK"; echo "  stack $STACK created"
fi
AWS_ROLE_ARN="$(aws cloudformation describe-stacks --stack-name "$STACK" --query 'Stacks[0].Outputs[?OutputKey==`RoleArn`].OutputValue' --output text)"

echo "▸ Register the BedrockAgentCore runtime 'aws-agentcore'"
arctl apply -f - <<EOF
apiVersion: ar.dev/v1alpha1
kind: Runtime
metadata: {name: aws-agentcore}
spec:
  type: BedrockAgentCore
  config: {roleArn: "${AWS_ROLE_ARN}", externalId: "${AWS_EXTERNAL_ID}", region: "${AWS_REGION}"}
EOF

echo "▸ Push the agent image to ECR (linux/amd64)"
ECR_HOST="${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_REGION}.amazonaws.com"; ECR_IMAGE="${ECR_HOST}/${AGENT}:0.0.1"
aws ecr describe-repositories --repository-names "$AGENT" >/dev/null 2>&1 || aws ecr create-repository --repository-name "$AGENT" >/dev/null
aws ecr get-login-password --region "$AWS_REGION" | docker login --username AWS --password-stdin "$ECR_HOST" >/dev/null
arctl build agentdemo --push --platform linux/amd64 --image "$ECR_IMAGE"

echo "▸ Push the agent source to git (AgentCore clones it)"
SLUG="${AGENT_GIT_URL#https://github.com/}"; SLUG="${SLUG%.git}"; BR="${AGENT_GIT_BRANCH:-main}"
CLONE_URL="https://x-access-token:$(gh auth token)@github.com/${SLUG}.git"
T="$(mktemp -d)"; cp -R agentdemo "$T/$AGENT"
( cd "$T" && git init -qb "$BR" && git add -A \
  && git -c user.email=demo@local -c user.name=demo commit -qm "agentdemo source" \
  && git remote add origin "$CLONE_URL" && git push -fq origin "$BR" )
rm -rf "$T"

echo "▸ Re-publish the agent (bedrock + ECR + git) and deploy it to AgentCore"
arctl apply -f - <<EOF
apiVersion: ar.dev/v1alpha1
kind: Agent
metadata: {name: ${AGENT}}
spec:
  description: Dice-rolling agent (roll_die, check_prime).
  modelName: us.anthropic.claude-haiku-4-5-20251001-v1:0
  modelProvider: bedrock
  source:
    image: ${ECR_IMAGE}
    repository: {url: ${CLONE_URL}, branch: ${BR}, subfolder: ${AGENT}}
EOF
arctl apply -f - <<EOF
apiVersion: ar.dev/v1alpha1
kind: Deployment
metadata: {name: ${AGENT}-agentcore}
spec:
  targetRef: {kind: Agent, name: ${AGENT}}
  runtimeRef: {kind: Runtime, name: aws-agentcore}
  runtimeConfig: {region: ${AWS_REGION}}
  env: {MODEL_PROVIDER: bedrock, AWS_REGION: ${AWS_REGION}}
EOF

echo "▸ Waiting for the AWS runtime (CREATING -> READY)"
for i in $(seq 1 25); do
  S="$(aws bedrock-agentcore-control list-agent-runtimes --region "$AWS_REGION" 2>/dev/null | jq -r '.agentRuntimes[]?|select(.agentRuntimeName=="agentdemo_agentcore")|.status')"
  echo "  [$i] agentdemo_agentcore: ${S:-<none>}"
  if echo "$S" | grep -qiE 'READY|FAILED'; then break; fi
  sleep 30
done
echo "AgentCore deploy submitted — test it with the ac-invoke cell below"

In [ ]:
./scripts/ac-invoke.sh "Roll a 13-sided die, add 5 to the result, then tell me if it is prime."

## 8. Run it from the kagent UI: traces and spans

Open the **kagent UI** (http://localhost:18007), invoke `agentdemo`, and watch the OTEL trace: the agent calls `roll_die`, then the MCP `sum` tool, then `check_prime`, span by span.

## 9. Govern the tools with an AccessPolicy (least privilege)

The `everything-server` exposes a sensitive `printenv` tool. A platform owner restricts the agent to only the tools it needs. `accesspolicy-on.sh` labels the MCP server for an **agentgateway waypoint** and applies a kagent **AccessPolicy** that denies `printenv`.

In [ ]:
# Least privilege: DENY the agent the `printenv` tool on the everything-server,
# enforced at an agentgateway waypoint. The other tools (sum/echo/...) keep working.
set -o pipefail
CTX=kind-$CLUSTER_NAME
MCP=demo-everything-server-agentdemo
AGENT=$(kubectl --context $CTX -n kagent get agents.kagent.dev -o name 2>/dev/null | sed 's#.*/##' | grep -i '^agentdemo' | head -1)
[ -n "$AGENT" ] || { echo "no agentdemo agent deployed on kagent"; exit 1; }
echo "agent=$AGENT  mcpserver=$MCP"

echo "▸ 1/3 route the MCP server through an agentgateway waypoint"
kubectl --context $CTX -n kagent label mcpserver $MCP kagent.solo.io/waypoint=true --overwrite
WP=mcpserver-$MCP-waypoint
for i in $(seq 1 30); do
  kubectl --context $CTX -n kagent get deploy $WP >/dev/null 2>&1 && break; sleep 2
done
kubectl --context $CTX -n kagent rollout status deploy/$WP --timeout=120s || true

echo "▸ 2/3 apply the AccessPolicy (a declarative CR) that denies the printenv tool"
kubectl --context $CTX -n kagent apply -f - <<EOF
apiVersion: policy.kagent-enterprise.solo.io/v1alpha1
kind: AccessPolicy
metadata:
  name: deny-printenv
  namespace: kagent
spec:
  from:
    subjects:
      - kind: Agent
        name: $AGENT
        namespace: kagent
  targetRef:
    kind: MCPServer
    name: $MCP
    tools:
      - printenv
  action: DENY
EOF
echo -n "AccessPolicy state: "; kubectl --context $CTX -n kagent get accesspolicy deny-printenv -o jsonpath='{.status.state}'; echo

echo "▸ 3/3 restart the agent so it re-lists tools through the waypoint"
kubectl --context $CTX -n kagent rollout restart deploy/$AGENT
kubectl --context $CTX -n kagent rollout status deploy/$AGENT --timeout=120s
echo "AccessPolicy enforcing — printenv is now denied"

The policy is a declarative custom resource — show it on the CLI:

In [ ]:
kubectl --context kind-$CLUSTER_NAME -n kagent get accesspolicy deny-printenv -o yaml

Ask the agent for its tools again (or re-ask in the kagent UI) — `printenv` is gone; `sum` and the rest still work:

In [ ]:
./scripts/ask.sh "List the exact names of every tool you can call. Output only a comma-separated list, nothing else."

## Reset / teardown

```sh
./scripts/accesspolicy-off.sh   # restore the full tool list
./scripts/reset.sh              # back to start (clears agentdemo, deployments)
./scripts/cleanup.sh            # full teardown (cluster, daemon, registry, AWS)
```